# GCP Pose Estimation — Kaggle Training

## Before running this notebook, do these 3 things on your laptop:

**1. Export your Google Drive cookies**
- Install Chrome extension **"Get cookies.txt LOCALLY"** from the Chrome Web Store
- Open a new tab → go to `drive.google.com` (logged in as `jacobantonynedu@gmail.com`)
- Click the extension icon → export → saves `cookies.txt` to Downloads

**2. Upload cookies as a private Kaggle dataset**
- kaggle.com → your profile → **Datasets** → **New Dataset**
- Upload `cookies.txt` → title: `gcp-drive-cookies` → visibility: **Private** → Create

**3. Add that dataset as input to this notebook**
- In this notebook → **Add Input** (top right) → Your Datasets → `gcp-drive-cookies` → Add

Then run cells top to bottom. GPU T4×2 + Internet must both be ON.

In [ ]:
# 1. Clone repo + install dependencies
!git clone https://github.com/J4KE-B/skylark-gcp gcp
%cd gcp
!pip -q install -r requirements.txt

In [ ]:
# 2. Download data using Google auth cookies
# cookies.txt must be added as a Kaggle input dataset (see markdown cell above)
import os, shutil, yaml, glob, json

# Locate the uploaded cookies file
cookie_files = glob.glob('/kaggle/input/**/cookies.txt', recursive=True)
assert cookie_files, (
    "cookies.txt not found in /kaggle/input/\n"
    "Fix: Add Input -> Your Datasets -> gcp-drive-cookies"
)
cookie_file = cookie_files[0]
print(f"Using cookies: {cookie_file}")

os.makedirs('data', exist_ok=True)

# Download the full GCP_Assignment_Datasets folder (train + test + labels)
# --cookies bypasses individual-file permission restrictions
print("Downloading data — takes 10-30 min...")
os.system(f'gdown --cookies "{cookie_file}" --folder 1RDNiAO1EynKrRDomcVNXQW1-ct5zzvE5 -O data/ --remaining-ok')

# gdown --folder may create data/<drive-folder-name>/ — flatten it
candidates = [d for d in os.listdir('data')
              if os.path.isdir(f'data/{d}') and d not in ('train_dataset', 'test_dataset')]
if candidates:
    src = f'data/{candidates[0]}'
    print(f"Moving {src}/* -> data/")
    for item in os.listdir(src):
        dst = f'data/{item}'
        if not os.path.exists(dst):
            shutil.move(f'{src}/{item}', dst)
    if not os.listdir(src):
        os.rmdir(src)

print("data/ contents:", sorted(os.listdir('data')))
assert os.path.exists('data/gcp_marks.json'), 'FAIL: gcp_marks.json missing'
assert os.path.isdir('data/train_dataset'),   'FAIL: train_dataset/ missing'
assert os.path.isdir('data/test_dataset'),    'FAIL: test_dataset/ missing'

# Wire config paths
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'] = {'label_file': 'data/gcp_marks.json',
                'train_dir':  'data/train_dataset',
                'test_dir':   'data/test_dataset',
                'output_dir': 'outputs'}
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))

# Verify how many labelled images actually landed on disk
labels = json.load(open('data/gcp_marks.json'))
found  = sum(1 for k in labels if os.path.exists(f"data/train_dataset/{k}"))
pct    = 100 * found // len(labels)
print(f"Train images on disk: {found}/{len(labels)} ({pct}%)")
if found < len(labels) * 0.8:
    print("WARNING: <80% of training images present — cookies may be stale or permissions wrong.")
else:
    print("OK — sufficient data to train.")

In [ ]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

In [ ]:
# 4. FULL run: restore 40 epochs
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 40
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

In [ ]:
# 5. Final predictions + download artifacts
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

## Artifacts to download

From the Kaggle Output panel (right sidebar), download these three files:

- `gcp/outputs/best.pt` — model weights (upload to Drive, copy the shareable link)
- `gcp/outputs/training_log.csv` — open it, read the last row for PCK@10/25/50 + Macro-F1
- `gcp/predictions.json` — this is your submission file for Skylark

Then update `README.md` in the repo:
1. Fill in the Results table with the PCK/F1 numbers
2. Add the Drive link for `best.pt`
3. `git add README.md && git commit -m "docs: add training results and weights link" && git push`